# Quantum Digital Signature, Rebuilt for Beginners

A step-by-step teaching notebook based on the implementation in this repository.


## Purpose

This notebook is a new, beginner-friendly companion to the repository's original notebook.

It has one central goal: help a first-year college student understand **what the code is doing**, **what the mathematics means**, and **why the protocol works at all**.

We will move in a very patient order:

1. Start with digital signatures in plain language.
2. Review the quantum ideas we need from scratch.
3. Build the quantum digital signature protocol at the story level.
4. Derive the key equations one step at a time.
5. Connect every important equation to the repository's code.
6. Run the full implementation and interpret the outputs carefully.


## What This Notebook Assumes

You do **not** need prior background in:

- Dirac notation
- quantum gates
- the SWAP test
- the Holevo bound
- digital signature protocols

You only need:

- basic algebra
- comfort with binary numbers
- willingness to go slowly

Whenever a symbol appears, we will define it nearby.


## How To Use This Notebook

Read it from top to bottom and run each cell in order.

The notebook is self-contained, but it is also faithful to this repository:

- the original README was studied
- the original notebook was read fully
- the same core class and protocol flow are preserved
- the original notebook remains untouched

> Teaching promise:
>
> If a formula appears, we will explain where it comes from before we use it.


---
## 1. Why Do We Care About Digital Signatures?


### Digital Signature in Plain Language

A digital signature is a way for a sender to attach proof that:

- the message really came from them
- the message was not changed afterward

In ordinary life, a handwritten signature helps with authenticity.
In computing, we need a mathematical version of the same idea.


### Why Classical Digital Signatures Matter

Classical digital signatures are used everywhere:

- secure software updates
- banking and payments
- signed contracts
- identity verification
- secure web connections

Usually, classical schemes depend on a hard mathematical problem.
For example, the security might rely on the assumption that some computation is infeasible for an attacker.


### Why Quantum Digital Signatures Are Interesting

A quantum digital signature tries to get security from **physics** rather than from a merely hard computation.

Two important quantum ideas appear in the repository:

- **quantum states cannot be copied perfectly** if they are unknown
- **quantum measurements cannot reveal unlimited classical information**

That is why this subject is interesting: instead of saying "forging is hard," we aim to say "forging is limited by the structure of quantum theory itself."


### The Specific Problem Solved Here

This repository implements a teaching version of the **Gottesman-Chuang quantum digital signature scheme**.

The idea is:

- Alice secretly chooses classical bit strings
- each secret string is converted into a quantum state
- those quantum states act like public keys
- later, Alice reveals the appropriate secret strings to sign a message
- Bob or Charlie check whether the revealed strings match the stored quantum public keys


---
## 2. Quantum Prerequisites in Plain Language


### Bits vs Qubits

A **classical bit** can be only one of two values:

$$
0 \quad \text{or} \quad 1.
$$

A **qubit** is different.

A qubit can be in a state that combines the two basis possibilities:

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle.
$$

Here:

- $|0\rangle$ and $|1\rangle$ are the basic reference states
- $\alpha$ and $\beta$ are called **amplitudes**


### The Basis States $|0\rangle$ and $|1\rangle$

In this notebook, the two basic one-qubit states are written as

$$
|0\rangle = \begin{pmatrix}1 \\ 0\end{pmatrix},
\qquad
|1\rangle = \begin{pmatrix}0 \\ 1\end{pmatrix}.
$$

These are called the **computational basis states**.

A measurement in the computational basis asks:

- "Do I see $0$?"
- or "Do I see $1$?"


### Superposition

The expression

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle
$$

is called a **superposition**.

It does **not** mean "the qubit is secretly either $0$ or $1$ and we just do not know which."
It means the state genuinely carries both components at once until measurement.


### Probability Amplitudes

The numbers $\alpha$ and $\beta$ are not probabilities directly.
They are **probability amplitudes**.

For a valid one-qubit state, they must satisfy

$$
|\alpha|^2 + |\beta|^2 = 1.
$$

The squares of their magnitudes become probabilities.


### Measurement Probabilities

If

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

then a computational-basis measurement gives

$$
P(0) = |\alpha|^2,
\qquad
P(1) = |\beta|^2.
$$

In the state family used in this repository, the amplitudes are real numbers, so we will often write

$$
P(0) = \alpha^2,
\qquad
P(1) = \beta^2.
$$


### A Very Short Introduction to Bra-Ket Notation

Dirac notation uses:

- a **ket** $|\psi\rangle$ for a column vector
- a **bra** $\langle \psi|$ for the corresponding row vector with complex conjugation

For real amplitudes, the conjugation step does nothing, so

$$
|\psi\rangle =
\begin{pmatrix}
\alpha \\
\beta
\end{pmatrix}
\quad \Longrightarrow \quad
\langle \psi| =
\begin{pmatrix}
\alpha & \beta
\end{pmatrix}.
$$

This notation becomes very convenient when we compute overlaps such as $\langle a|b\rangle$.


### Matrix-Vector Viewpoint

Quantum gates are matrices.
Quantum states are vectors.

So a gate acting on a state is just matrix multiplication.

If a gate is called $U$, then

$$
|\psi_{\text{new}}\rangle = U |\psi_{\text{old}}\rangle.
$$

That simple sentence is enough to understand a large fraction of this notebook.


### One More Helpful Fact for This Notebook

The repository uses the gate $R_Y(\phi)$, a rotation around the $y$ axis.

Because the notebook starts from $|0\rangle$ and uses only $R_Y$ in the key-state construction, the resulting amplitudes stay real.

That is helpful for beginners because the states look like

$$
\cos(\cdot)\,|0\rangle + \sin(\cdot)\,|1\rangle,
$$

not a more complicated complex-amplitude expression.


### Mini Quiz 1

Try to answer before scrolling:

1. If a qubit is in $|0\rangle$, what is the probability of measuring $1$?
2. If a qubit is in $\frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$, what is $P(0)$?
3. Why are amplitudes not the same thing as probabilities?

Short answers:

1. $0$
2. $\frac{1}{2}$
3. Because probabilities come from squared magnitudes of amplitudes.


---
## 3. Imports and Small Helper Tools


The next cell imports the same main libraries used by the repository:

- `numpy` for numerical work
- `matplotlib` for plots
- `qiskit` and `qiskit-aer` for quantum circuits and simulation

I also define a few tiny helper functions so the later cells can focus on ideas instead of string formatting.


In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from typing import Dict, List, Tuple

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(suppress=True)

print("=" * 60)
print("Library versions")
print("=" * 60)
import qiskit, qiskit_aer, scipy
print(f"qiskit      : {qiskit.__version__}")
print(f"qiskit-aer  : {qiskit_aer.__version__}")
print(f"numpy       : {np.__version__}")
print(f"scipy       : {scipy.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print("=" * 60)


In [ ]:
def bit_array_to_string(bits) -> str:
    return "".join(str(int(b)) for b in bits)


def bit_string_to_int(bit_string: str) -> int:
    return int(bit_string, 2)


def amplitudes_from_j(j: int, theta: float) -> Tuple[float, float]:
    alpha = float(np.cos(j * theta))
    beta = float(np.sin(j * theta))
    return alpha, beta


def amplitudes_from_key(key_bits, theta: float) -> Tuple[str, int, float, float]:
    if isinstance(key_bits, str):
        bit_string = key_bits
    else:
        bit_string = bit_array_to_string(key_bits)
    j = bit_string_to_int(bit_string)
    alpha, beta = amplitudes_from_j(j, theta)
    return bit_string, j, alpha, beta


def format_state(alpha: float, beta: float, decimals: int = 3) -> str:
    return f"{alpha:+.{decimals}f}|0> {beta:+.{decimals}f}|1>"


def overlap_from_indices(j_a: int, j_b: int, theta: float) -> float:
    return float(np.cos((j_a - j_b) * theta))


def swap_probability_from_overlap(overlap: float) -> float:
    return float((1.0 + abs(overlap) ** 2) / 2.0)


print("Helper functions ready.")


---
## 4. The Protocol at the Big-Picture Level


### Who Are the Main Characters?

The usual story uses three people:

- **Alice**: the signer
- **Bob**: a recipient
- **Charlie**: another recipient or judge

Alice wants to send a signed message.
Bob wants to know whether it is genuine.
Charlie matters because a useful signature should still make sense when Bob forwards it.


### The Classical One-Time Signature Idea Behind the Quantum Version

Before adding any quantum mechanics, there is a simple classical pattern:

- for each possible message bit value, Alice prepares a secret
- the corresponding public information is shared ahead of time
- later, to sign bit $0$, Alice reveals the secret for bit $0$
- to sign bit $1$, Alice reveals the secret for bit $1$

The quantum protocol in this repository follows that same spirit.


### The Story Version of This Repository's Protocol

For each message position:

1. Alice chooses two secret classical keys, one meant for signing `0` and one for signing `1`.
2. She converts each classical key into a quantum state.
3. Those quantum states play the role of public keys.
4. Recipients compare public-key copies using the SWAP test.
5. To sign the actual message, Alice reveals the matching classical keys.
6. The recipient rebuilds the expected quantum state from each revealed key and checks whether it matches the stored public key.


### The Phases We Will See in Code

The implementation naturally breaks into five phases:

- key generation
- public-key consistency checking
- signature generation
- signature verification
- security-parameter exploration


### Notation Dictionary

To keep symbols from becoming confusing, I will use:

- $L$: number of bits in one private key
- $M$: number of key pairs per message position
- $n$: number of qubits in one public-key state
- $t$: message position index
- $i$: copy index
- $k_0^{(t,i)}$: secret key used if message bit at position $t$ is $0$
- $k_1^{(t,i)}$: secret key used if message bit at position $t$ is $1$

In the code, the message-position index is called `idx_b`.


### Mini Quiz 2

Suppose Alice wants to sign the bit `1`.

Which secret should she reveal?

- the secret associated with bit `1`
- not the secret associated with bit `0`

That simple rule becomes the heart of the signing step.


---
## 5. Mathematical Foundations


### Step 1: Turn a Bit String Into an Integer

A private key is a classical bit string of length $L$:

$$
k = k_{L-1}k_{L-2}\cdots k_1k_0,
\qquad
k_r \in \{0,1\}.
$$

The repository converts that bit string into an integer

$$
j = \operatorname{int}(k, 2).
$$

The `2` means "interpret this string in base 2."


### Where Does That Integer Formula Come From?

If

$$
k = b_{L-1}b_{L-2}\cdots b_1b_0,
$$

then its value in binary is

$$
j = b_{L-1}2^{L-1} + b_{L-2}2^{L-2} + \cdots + b_1 2^1 + b_0 2^0.
$$

Example:

$$
1011_2 = 1\cdot 2^3 + 0\cdot 2^2 + 1\cdot 2^1 + 1\cdot 2^0 = 8 + 0 + 2 + 1 = 11.
$$

That integer $j$ will decide how far we rotate the qubit.


In [ ]:
bit_string_examples = ["00", "01", "10", "11", "1011", "101100"]

print("Binary string  ->  integer j")
print("-" * 32)
for bit_string in bit_string_examples:
    print(f"{bit_string:>10}  ->  {bit_string_to_int(bit_string):>3d}")


### Step 2: Why Is $\theta = \pi / 2^L$?

Once a key becomes an integer $j$, we want to map it to an angle.

There are $2^L$ possible keys of length $L$.
So it is natural to split an angular interval into $2^L$ equally spaced pieces.

The repository chooses

$$
\theta = \frac{\pi}{2^L}.
$$

Then the integer $j$ gets mapped to the angle

$$
j\theta.
$$


### Why This Choice Is Reasonable

The possible values of $j$ are

$$
0,1,2,\dots,2^L-1.
$$

So the possible angles are

$$
0,\theta,2\theta,\dots,(2^L-1)\theta.
$$

Because

$$
(2^L - 1)\theta = (2^L - 1)\frac{\pi}{2^L} = \pi - \frac{\pi}{2^L},
$$

the states spread across an interval that is almost the whole range from $0$ to $\pi$.
That gives many nearby but not identical states.


In [ ]:
print("L   number of keys   theta = pi/2^L      delta = cos(theta)")
print("-" * 62)
for L_value in range(2, 8):
    theta_value = np.pi / (2 ** L_value)
    delta_value = np.cos(theta_value)
    print(
        f"{L_value:1d}   {2**L_value:13d}   "
        f"{theta_value:14.6f}   {delta_value:16.6f}"
    )


### Step 3: The Matrix Form of the $R_Y$ Gate

The repository uses a single-qubit rotation gate:

$$
R_Y(\phi).
$$

Its matrix is

$$
R_Y(\phi)=
\begin{pmatrix}
\cos(\phi/2) & -\sin(\phi/2) \\
\sin(\phi/2) & \cos(\phi/2)
\end{pmatrix}.
$$

Here $\phi$ is the rotation angle supplied to the gate.


### Derivation: Apply $R_Y(\phi)$ to $|0\rangle$

Start from

$$
|0\rangle = \begin{pmatrix}1 \\ 0\end{pmatrix}.
$$

Then

$$
R_Y(\phi)|0\rangle
=
\begin{pmatrix}
\cos(\phi/2) & -\sin(\phi/2) \\
\sin(\phi/2) & \cos(\phi/2)
\end{pmatrix}
\begin{pmatrix}
1 \\
0
\end{pmatrix}.
$$

Multiply the matrix by the vector row by row.


### Finish the Multiplication Carefully

The top entry becomes

$$
\cos(\phi/2)\cdot 1 + \bigl(-\sin(\phi/2)\bigr)\cdot 0 = \cos(\phi/2).
$$

The bottom entry becomes

$$
\sin(\phi/2)\cdot 1 + \cos(\phi/2)\cdot 0 = \sin(\phi/2).
$$

Therefore

$$
R_Y(\phi)|0\rangle
=
\begin{pmatrix}
\cos(\phi/2) \\
\sin(\phi/2)
\end{pmatrix}
=
\cos(\phi/2)|0\rangle + \sin(\phi/2)|1\rangle.
$$


### Now Set $\phi = 2j\theta$

The repository chooses the gate angle

$$
\phi = 2j\theta.
$$

Substitute this into the formula above:

$$
R_Y(2j\theta)|0\rangle
= \cos\!\left(\frac{2j\theta}{2}\right)|0\rangle
+ \sin\!\left(\frac{2j\theta}{2}\right)|1\rangle.
$$

Since $\frac{2j\theta}{2}=j\theta$, we get

$$
|f_k\rangle
= R_Y(2j\theta)|0\rangle
= \cos(j\theta)|0\rangle + \sin(j\theta)|1\rangle.
$$

This is the key state family used by the repository.


### Plain-Language Interpretation

The private key $k$ does not directly become a probability.
It becomes an integer $j$, and that integer picks a rotation angle.

So the private key chooses a point on a smooth family of quantum states.

Nearby keys produce nearby angles.
Nearby angles produce nearby quantum states.
That is exactly why distinguishing keys is not trivial.


In [ ]:
L_demo = 3
theta_demo = np.pi / (2 ** L_demo)
demo_keys = ["000", "001", "010", "011", "100", "101", "110", "111"]

print(f"For L = {L_demo}, theta = pi / 2^{L_demo} = {theta_demo:.6f}")
print()
print("key    j    amplitudes of |f_k>")
print("-" * 46)
for key in demo_keys:
    _, j_value, alpha, beta = amplitudes_from_key(key, theta_demo)
    print(f"{key}   {j_value:1d}    {format_state(alpha, beta, decimals=4)}")


---
## 6. Quantum One-Way Function


### What Does "Easy to Compute" Mean Here?

If someone gives us the classical key $k$, then:

1. convert $k$ to the integer $j$
2. compute the angle $2j\theta$
3. apply the gate $R_Y(2j\theta)$ to $|0\rangle$

Each step is straightforward and efficient.

So the map

$$
k \longmapsto |f_k\rangle
$$

is easy to evaluate.


### What Does "Hard to Invert" Mean Here?

Inversion would mean:

- you are given one or more copies of the quantum state $|f_k\rangle$
- you try to recover the original classical key $k$

The problem is that the quantum state does not let you read out all $L$ bits freely.
Measurements give limited information, and measuring the state generally disturbs it.


### Tiny Example With $L=2$

If $L=2$, then there are $2^2=4$ possible keys:

$$
00,\ 01,\ 10,\ 11.
$$

Their associated angles are

$$
0,\ \frac{\pi}{4},\ \frac{\pi}{2},\ \frac{3\pi}{4}.
$$

So the corresponding states are:

- $00 \mapsto |0\rangle$
- $01 \mapsto \tfrac{1}{\sqrt{2}}|0\rangle + \tfrac{1}{\sqrt{2}}|1\rangle$
- $10 \mapsto |1\rangle$
- $11 \mapsto -\tfrac{1}{\sqrt{2}}|0\rangle + \tfrac{1}{\sqrt{2}}|1\rangle$


### Why This Is Useful for Signatures

The protocol wants a public object that others can verify, but cannot fully reverse.

A classical public key is a string.
Here the public key is a quantum state.

The state still allows checks, but it does not hand over the whole classical secret for free.


In [ ]:
L_tiny = 2
theta_tiny = np.pi / (2 ** L_tiny)
tiny_keys = ["00", "01", "10", "11"]

print(f"Tiny example with L = {L_tiny}")
print(f"theta = {theta_tiny:.6f} rad")
print()
for key in tiny_keys:
    _, j_value, alpha, beta = amplitudes_from_key(key, theta_tiny)
    print(
        f"key {key} -> j = {j_value} -> "
        f"|f_k> = {format_state(alpha, beta, decimals=4)}"
    )


---
## 7. Holevo Bound Intuition


### First the Intuition

A quantum state can depend on continuous parameters.
That does **not** mean we can read out unlimited classical information from one copy of that state.

Measurement outcomes are limited.

This is one of the deep lessons of quantum information theory:

- a quantum system can be rich
- but the amount of classical information you can extract from it is still bounded


### Simple Form of the Holevo Bound Used Here

In this repository, each public key is an $n$-qubit state.

If an attacker gets $M$ copies of that state, then the total number of qubits available is $nM$.

The Holevo bound says that the accessible classical information is at most on the order of

$$
nM \text{ bits}.
$$

Meanwhile, the secret key itself is an $L$-bit classical string.


### What "Information Leakage" Means

Leakage does **not** mean the attacker learns nothing.

It means:

- some information can leak from the public quantum states
- but not enough to fully reveal a long secret key when $L$ is much larger than $nM$

So the public quantum states are informative enough to verify, but not informative enough to fully decode the private secret.


### Why More Copies Help an Attacker, But Not Infinitely

If an attacker gets more copies of the public state, then they can perform more measurements.
So more copies usually means more information.

But that extra information scales with the number of qubits they physically hold.

That is why the rough design rule is

$$
L - nM \gg 1.
$$

The secret should be much longer than the amount of classical information the attacker could hope to extract.


### A Careful Honesty Note

The Holevo bound gives an information-theoretic limit.
It does **not**, by itself, automatically provide every detailed forging probability formula.

So in this notebook we will use it honestly:

- as the core intuition for why inversion is limited
- not as a magical one-line proof of every security claim


---
## 8. Overlap and Distinguishability


### Why Overlap Matters

If two quantum states are identical, no test should separate them.
If they are perfectly orthogonal, they can be distinguished ideally.
Most states in this notebook are in between.

The mathematical quantity that captures this is the **inner product**:

$$
\langle f_k | f_{k'} \rangle.
$$


### Write the Two States Explicitly

Let

$$
|f_k\rangle = \cos(j\theta)|0\rangle + \sin(j\theta)|1\rangle
$$

and

$$
|f_{k'}\rangle = \cos(j'\theta)|0\rangle + \sin(j'\theta)|1\rangle.
$$

Because the amplitudes are real, the corresponding bra is

$$
\langle f_k|
=
\cos(j\theta)\langle 0| + \sin(j\theta)\langle 1|.
$$


### Derive the Inner Product Step by Step

Multiply bra and ket:

$$
\langle f_k|f_{k'}\rangle
=
\bigl(\cos(j\theta)\langle 0| + \sin(j\theta)\langle 1|\bigr)
\bigl(\cos(j'\theta)|0\rangle + \sin(j'\theta)|1\rangle\bigr).
$$

Expanding gives four terms:

$$
\cos(j\theta)\cos(j'\theta)\langle 0|0\rangle
+ \cos(j\theta)\sin(j'\theta)\langle 0|1\rangle
+ \sin(j\theta)\cos(j'\theta)\langle 1|0\rangle
+ \sin(j\theta)\sin(j'\theta)\langle 1|1\rangle.
$$

Now use the basis relations

$$
\langle 0|0\rangle = 1,\quad
\langle 1|1\rangle = 1,\quad
\langle 0|1\rangle = 0,\quad
\langle 1|0\rangle = 0.
$$


### Simplify the Expression

The cross terms vanish, so we get

$$
\langle f_k|f_{k'}\rangle
=
\cos(j\theta)\cos(j'\theta) + \sin(j\theta)\sin(j'\theta).
$$

Use the trigonometric identity

$$
\cos A \cos B + \sin A \sin B = \cos(A-B).
$$

Therefore

$$
\langle f_k|f_{k'}\rangle
= \cos\bigl((j-j')\theta\bigr).
$$


### What Is $\delta$?

The repository uses

$$
\delta = \cos(\theta) = \cos\!\left(\frac{\pi}{2^L}\right).
$$

Why?

For two different keys, the smallest nonzero value of $|j-j'|$ is $1$.
On the interval from $\theta$ to $\pi-\theta$, the largest possible absolute value of the cosine occurs at the endpoints.

So for distinct keys,

$$
|\langle f_k|f_{k'}\rangle|
=
\left|\cos\bigl((j-j')\theta\bigr)\right|
\le \cos(\theta)=\delta.
$$

In plain language:

- $\delta$ measures the worst-case similarity between two different key states
- smaller $\delta$ means better distinguishability


In [ ]:
L_overlap = 3
theta_overlap = np.pi / (2 ** L_overlap)
compare_pairs = [("000", "001"), ("000", "010"), ("001", "011"), ("010", "111")]

print(f"Overlap examples for L = {L_overlap}, theta = {theta_overlap:.6f}")
print("-" * 68)
for key_a, key_b in compare_pairs:
    _, j_a, _, _ = amplitudes_from_key(key_a, theta_overlap)
    _, j_b, _, _ = amplitudes_from_key(key_b, theta_overlap)
    overlap = overlap_from_indices(j_a, j_b, theta_overlap)
    print(
        f"{key_a} vs {key_b} -> j difference = {j_a - j_b:+2d} -> "
        f"<f_k|f_k'> = {overlap:+.6f}"
    )


---
## 9. The SWAP Test


### Why Do We Need the SWAP Test?

In this protocol, recipients need to compare quantum public-key states.

But quantum states are not classical strings.
You cannot just "look at them" and compare every entry.

The SWAP test gives a clever probabilistic answer to the question:

> "How similar are these two quantum states?"


### The Circuit Idea

The SWAP test uses:

- one ancilla qubit
- one register holding $|a\rangle$
- one register holding $|b\rangle$

The steps are:

1. put the ancilla into superposition with a Hadamard gate
2. use a controlled-SWAP so the ancilla decides whether the two states swap places
3. apply another Hadamard to the ancilla
4. measure the ancilla


In [ ]:
theta_swap_demo = np.pi / 4

state_a = QuantumCircuit(1, name="|a>")
state_a.ry(2 * (1 * theta_swap_demo), 0)

state_b = QuantumCircuit(1, name="|b>")
state_b.ry(2 * (2 * theta_swap_demo), 0)

swap_demo = QuantumCircuit(3, 1)
swap_demo.h(0)
swap_demo.compose(state_a, qubits=[1], inplace=True)
swap_demo.compose(state_b, qubits=[2], inplace=True)
swap_demo.cswap(0, 1, 2)
swap_demo.h(0)
swap_demo.measure(0, 0)

swap_demo.draw("mpl", fold=-1)


### Derivation: State After the First Hadamard

Start with

$$
|0\rangle|a\rangle|b\rangle.
$$

Applying a Hadamard gate to the ancilla gives

$$
\frac{|0\rangle + |1\rangle}{\sqrt{2}} \otimes |a\rangle \otimes |b\rangle.
$$

Distribute the tensor product:

$$
|\Psi_0\rangle
=
\frac{1}{\sqrt{2}}
\bigl(
|0\rangle|a\rangle|b\rangle
+
|1\rangle|a\rangle|b\rangle
\bigr).
$$


### Derivation: After the Controlled-SWAP

The controlled-SWAP does nothing when the ancilla is $|0\rangle$.
It swaps the two target registers when the ancilla is $|1\rangle$.

So the state becomes

$$
|\Psi_1\rangle
=
\frac{1}{\sqrt{2}}
\bigl(
|0\rangle|a\rangle|b\rangle
+
|1\rangle|b\rangle|a\rangle
\bigr).
$$


### Derivation: After the Final Hadamard

Use

$$
H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}},
\qquad
H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}.
$$

Apply $H$ to the ancilla in $|\Psi_1\rangle$:

$$
|\Psi_2\rangle
=
\frac{1}{2}
\Bigl[
|0\rangle\bigl(|a\rangle|b\rangle + |b\rangle|a\rangle\bigr)
+
|1\rangle\bigl(|a\rangle|b\rangle - |b\rangle|a\rangle\bigr)
\Bigr].
$$

The amplitude attached to ancilla state $|0\rangle$ is therefore

$$
\frac{1}{2}\bigl(|a\rangle|b\rangle + |b\rangle|a\rangle\bigr).
$$


### Compute the Probability of Measuring Ancilla $0$

The probability is the squared norm of that ancilla-$0$ part:

$$
P(0)
=
\left\|
\frac{1}{2}\bigl(|a\rangle|b\rangle + |b\rangle|a\rangle\bigr)
\right\|^2.
$$

Expanding the norm gives

$$
P(0)
=
\frac{1}{4}
\left(
\langle a|\langle b| + \langle b|\langle a|
\right)
\left(
|a\rangle|b\rangle + |b\rangle|a\rangle
\right).
$$

After simplifying,

$$
P(0)=\frac{1+|\langle a|b\rangle|^2}{2}.
$$


### Plain-Language Interpretation

This formula says:

- if the states are identical, then $|\langle a|b\rangle|=1$ and $P(0)=1$
- if the states are orthogonal, then $|\langle a|b\rangle|=0$ and $P(0)=\tfrac{1}{2}$
- if they are partly similar, the probability sits somewhere in between

So the SWAP test turns "state similarity" into "probability of seeing ancilla 0."


In [ ]:
theoretical_examples = [
    ("identical", 1.0),
    ("partly overlapping", 0.80),
    ("orthogonal", 0.0),
]

print("Example overlap  ->  SWAP-test P(ancilla = 0)")
print("-" * 49)
for label, overlap in theoretical_examples:
    print(f"{label:18s} -> {swap_probability_from_overlap(overlap):.4f}")


---
## 10. Mapping the Repository Before We Reimplement It


### What Was Studied in the Repository

This repository is small, so it was possible to inspect everything directly:

- `README.md`
- `quantum_digital_signature.ipynb`
- the four generated figures:
  `example_circuit.png`,
  `swap_test_comparison.png`,
  `verification_decision.png`,
  `forgery_analysis.png`

The original notebook contains one central class:

- `QuantumDigitalSignature`

with these major methods:

- `__init__`
- `_generate_quantum_state`
- `generate_keys`
- `swap_test`
- `verify_key_pair`
- `generate_signature`
- `verify_signature`


### Important Implementation Simplifications

The repository is a teaching simulation, not a full deployment system.

Some important simplifications are:

- the demo uses one-qubit public-key states (`n_qubits = 1`)
- the public keys are represented by circuits on a simulator
- public-key equality is demonstrated by directly running the SWAP test on stored circuits
- the message-level verification aggregates failures across the message and keeps the repository's threshold rule for demonstration

I will preserve the repository's behavior so the notebook matches the code, while clearly pointing out where the literature-level protocol is richer.


### What the Next Code Cell Will Do

The next code cell defines a cleaned-up, reproducible version of the same core class.

The protocol logic stays the same, but I add three teaching conveniences:

- a random seed for reproducibility
- helper methods such as `key_to_int` and `key_to_angle`
- a `per_bit_failures` summary to make the verification output easier to interpret


### Method Walkthrough: `__init__`

Purpose:

- store the parameters $n$, $L$, $M$, and $\delta$
- compute the spacing angle $\theta = \pi/2^L$
- create a simulator
- create empty containers for private keys and public-key circuits

Code connection:

- `self.THETA` is the exact code version of the formula for $\theta$
- `self.private_keys` and `self.public_keys` hold the protocol data generated later


### Method Walkthrough: `_generate_quantum_state`

Purpose:

- take a classical key $k$
- convert it to the integer $j$
- prepare the quantum state

Mathematics:

$$
|f_k\rangle = R_Y(2j\theta)|0\rangle
= \cos(j\theta)|0\rangle + \sin(j\theta)|1\rangle.
$$

Code connection:

- `int(bit_string, 2)` computes $j$
- `qc.ry(2.0 * angle, 0)` implements the $R_Y(2j\theta)$ gate


### Method Walkthrough: `generate_keys`

Purpose:

- for each message position
- generate $M$ random pairs of classical keys
- build the matching public-key circuits

Output:

- `private_keys[idx_b][i] = (k0, k1)`
- `public_keys[idx_b][i] = (circuit_for_k0, circuit_for_k1)`


### Method Walkthrough: `swap_test`

Purpose:

- compare two public-key circuits
- estimate how similar their output states are

Mathematics:

$$
P(\text{ancilla}=0)=\frac{1+|\langle a|b\rangle|^2}{2}.
$$

Code connection:

- `qc.h(0)` creates ancilla superposition
- `qc.cswap(...)` performs the Fredkin gate
- measuring the ancilla estimates the above probability


### Method Walkthrough: `verify_key_pair`

Purpose:

- pick the public-key circuit corresponding to the actual message bit
- compare two copies with the SWAP test

Why it matters:

In the protocol story, Bob and Charlie want confidence that the public keys they received are consistent copies of the same state.


### Method Walkthrough: `generate_signature`

Purpose:

- reveal only the private keys associated with the actual message bits

Example:

- if the message bit is `0`, reveal the `k0` keys
- if the message bit is `1`, reveal the `k1` keys

The opposite keys stay secret.


### Method Walkthrough: `verify_signature`

Purpose:

- rebuild the expected public-key state from each revealed private key
- apply the inverse of that rebuilt circuit to the stored public-key state
- measure whether the result returns to $|0\cdots 0\rangle$

Honest case:

$$
R_Y(-2j\theta)R_Y(2j\theta)=I.
$$

Forged case:

$$
P(0)=\cos^2\bigl((j-j')\theta\bigr) < 1
$$

when $j\neq j'$.


In [ ]:
class QuantumDigitalSignature:
    """Beginner-friendly version of the repository's main QDS class."""

    IDENTITY_THRESHOLD: float = 0.85
    VERIFICATION_PASS_THRESHOLD: float = 0.90

    def __init__(
        self,
        n_qubits: int,
        L: int,
        M: int,
        delta: float | None = None,
        shots: int = 4096,
        seed: int | None = None,
    ) -> None:
        self.n_qubits = n_qubits
        self.L = L
        self.M = M
        self.delta = float(np.cos(np.pi / 2**L) if delta is None else delta)
        self.THETA = np.pi / float(2 ** L)
        self.SHOTS = shots
        self.seed = seed

        self.simulator = AerSimulator(seed_simulator=seed)
        self.rng = np.random.default_rng(seed)

        self.private_keys: Dict[int, List[Tuple[np.ndarray, np.ndarray]]] = {}
        self.public_keys: Dict[int, List[Tuple[QuantumCircuit, QuantumCircuit]]] = {}
        self.message_bits: str = ""

    def key_to_int(self, k: np.ndarray) -> int:
        return int(bit_array_to_string(k), 2)

    def key_to_angle(self, k: np.ndarray) -> float:
        return self.key_to_int(k) * self.THETA

    def theoretical_overlap(self, key_a: np.ndarray, key_b: np.ndarray) -> float:
        return overlap_from_indices(self.key_to_int(key_a), self.key_to_int(key_b), self.THETA)

    def _generate_quantum_state(self, k: np.ndarray) -> QuantumCircuit:
        k_str = bit_array_to_string(k)
        qc = QuantumCircuit(self.n_qubits, name="fk")

        if self.n_qubits == 1:
            j = int(k_str, 2)
            angle = float(j) * self.THETA
            qc.ry(2.0 * angle, 0)
        else:
            sub_len = max(1, self.L // self.n_qubits)
            theta_sub = np.pi / float(2 ** sub_len)
            for q in range(self.n_qubits):
                start = q * sub_len
                raw = k_str[start:start + sub_len] if start < len(k_str) else ""
                sub_k = raw.ljust(sub_len, "0")[:sub_len]
                j_q = int(sub_k, 2)
                angle_q = float(j_q) * theta_sub
                qc.ry(2.0 * angle_q, q)

        return qc

    def generate_keys(self, message_bits: str) -> Tuple[dict, dict]:
        self.message_bits = message_bits
        self.private_keys = {}
        self.public_keys = {}

        for idx_b in range(len(message_bits)):
            self.private_keys[idx_b] = []
            self.public_keys[idx_b] = []

            for _ in range(self.M):
                k0 = self.rng.integers(0, 2, size=self.L, dtype=int)
                k1 = self.rng.integers(0, 2, size=self.L, dtype=int)
                self.private_keys[idx_b].append((k0.copy(), k1.copy()))
                self.public_keys[idx_b].append(
                    (
                        self._generate_quantum_state(k0),
                        self._generate_quantum_state(k1),
                    )
                )

        return self.private_keys, self.public_keys

    def swap_test(
        self,
        circuit_a: QuantumCircuit,
        circuit_b: QuantumCircuit,
    ) -> Tuple[float, dict]:
        n = self.n_qubits
        qc = QuantumCircuit(2 * n + 1, 1)

        qc.h(0)
        qc.compose(circuit_a, qubits=list(range(1, n + 1)), inplace=True)
        qc.compose(circuit_b, qubits=list(range(n + 1, 2 * n + 1)), inplace=True)
        for i in range(n):
            qc.cswap(0, i + 1, n + i + 1)
        qc.h(0)
        qc.measure(0, 0)

        transpiled = transpile(qc, self.simulator, seed_transpiler=self.seed)
        job = self.simulator.run(transpiled, shots=self.SHOTS)
        counts = job.result().get_counts()
        p_zero = counts.get("0", 0) / self.SHOTS
        return p_zero, counts

    def verify_key_pair(self, idx_b: int, i: int, j: int) -> dict:
        if not self.message_bits:
            raise ValueError("Call generate_keys() before verify_key_pair().")

        bit_value = int(self.message_bits[idx_b])
        circ_i = self.public_keys[idx_b][i][bit_value]
        circ_j = self.public_keys[idx_b][j][bit_value]
        p_same, raw_counts = self.swap_test(circ_i, circ_j)
        return {
            "p_same": p_same,
            "are_identical": bool(p_same >= self.IDENTITY_THRESHOLD),
            "raw_counts": raw_counts,
        }

    def generate_signature(self, message_bits: str) -> dict:
        if not self.private_keys:
            raise ValueError("Call generate_keys() before generate_signature().")

        revealed: Dict[int, dict] = {}
        for idx_b, bit in enumerate(message_bits):
            bit_value = int(bit)
            revealed[idx_b] = {"bit_value": bit_value, "keys": []}
            for i in range(self.M):
                k0, k1 = self.private_keys[idx_b][i]
                chosen = k0 if bit_value == 0 else k1
                revealed[idx_b]["keys"].append(chosen.tolist())
        return {"message": message_bits, "signature": revealed}

    def verify_signature(self, signed_message: dict, c1: float, c2: float) -> dict:
        message_bits = signed_message["message"]
        signature = signed_message["signature"]
        total_verifications = len(message_bits) * self.M
        failed = 0
        per_bit_failures: Dict[int, int] = {}

        for idx_b in range(len(message_bits)):
            bit_value = int(message_bits[idx_b])
            sig_data = signature[idx_b]
            bit_failures = 0

            for i in range(self.M):
                revealed_k = np.array(sig_data["keys"][i], dtype=int)
                pub_circuit = self.public_keys[idx_b][i][bit_value]
                regen_circuit = self._generate_quantum_state(revealed_k)

                n = self.n_qubits
                ver_qc = QuantumCircuit(n, n)
                ver_qc.compose(pub_circuit, qubits=list(range(n)), inplace=True)
                ver_qc.compose(regen_circuit.inverse(), qubits=list(range(n)), inplace=True)
                ver_qc.measure(list(range(n)), list(range(n)))

                transpiled = transpile(ver_qc, self.simulator, seed_transpiler=self.seed)
                job = self.simulator.run(transpiled, shots=self.SHOTS)
                counts_v = job.result().get_counts()

                all_zeros = "0" * n
                p_zero = counts_v.get(all_zeros, 0) / self.SHOTS
                if p_zero < self.VERIFICATION_PASS_THRESHOLD:
                    failed += 1
                    bit_failures += 1

            per_bit_failures[idx_b] = bit_failures

        threshold_c1 = c1 * self.M
        threshold_c2 = c2 * self.M

        if failed <= threshold_c1:
            verdict = "1-ACC"
            verdict_full = "1-Accepted: signature is VALID and TRANSFERABLE"
        elif failed >= threshold_c2:
            verdict = "REJ"
            verdict_full = "Rejected: signature is INVALID"
        else:
            verdict = "0-ACC"
            verdict_full = "0-Accepted: signature is VALID but NOT TRANSFERABLE"

        return {
            "s_j": failed,
            "total_verifications": total_verifications,
            "threshold_c1": threshold_c1,
            "threshold_c2": threshold_c2,
            "verdict": verdict,
            "verdict_full": verdict_full,
            "failure_rate": failed / total_verifications if total_verifications else 0.0,
            "per_bit_failures": per_bit_failures,
        }


print("QuantumDigitalSignature class defined.")
print(f"IDENTITY_THRESHOLD          = {QuantumDigitalSignature.IDENTITY_THRESHOLD}")
print(f"VERIFICATION_PASS_THRESHOLD = {QuantumDigitalSignature.VERIFICATION_PASS_THRESHOLD}")


---
## 11. Instantiate the Scheme Used in the Repository Demo


The original notebook uses these main demo parameters:

- $n = 1$ qubit per public-key state
- $L = 6$ bits per private key
- $M = 5$ key pairs per message position
- message `101100101`

I keep those same values so the logic matches the repository.

I only add a seed so a beginner running the notebook can reproduce the same random keys and approximate counts.


In [ ]:
SEED = 7
N_QUBITS = 1
L = 6
M = 5
DELTA = float(np.cos(np.pi / 2**L))
MESSAGE_BITS = "101100101"

qds = QuantumDigitalSignature(
    n_qubits=N_QUBITS,
    L=L,
    M=M,
    delta=DELTA,
    shots=4096,
    seed=SEED,
)

print("=" * 60)
print("QDS initialisation")
print("=" * 60)
print(f"seed      : {SEED}")
print(f"n_qubits  : {qds.n_qubits}")
print(f"L         : {qds.L}  ({2**qds.L} possible keys)")
print(f"M         : {qds.M}")
print(f"delta     : {qds.delta:.6f}")
print(f"THETA     : {qds.THETA:.6f}")
print(f"SHOTS     : {qds.SHOTS}")
print(f"message   : {MESSAGE_BITS}")
print(f"length    : {len(MESSAGE_BITS)} bits")
print(f"L - n*M   : {L - N_QUBITS*M}")
print("=" * 60)


---
## 12. Key Generation


### What `generate_keys` Returns

For each message position, the code creates $M$ independent pairs:

$$
\left(k_0^{(t,1)}, k_1^{(t,1)}\right),
\left(k_0^{(t,2)}, k_1^{(t,2)}\right),
\dots,
\left(k_0^{(t,M)}, k_1^{(t,M)}\right).
$$

Each classical key is then converted into a public-key circuit.


In [ ]:
private_keys, public_keys = qds.generate_keys(MESSAGE_BITS)

print("Private keys (first 3 message positions, first 2 copies)")
print("=" * 72)
for idx_b in range(min(3, len(MESSAGE_BITS))):
    print(f"message position {idx_b} (bit value = {MESSAGE_BITS[idx_b]})")
    for i in range(min(2, M)):
        k0, k1 = private_keys[idx_b][i]
        j0 = qds.key_to_int(k0)
        j1 = qds.key_to_int(k1)
        print(
            f"  copy {i}: "
            f"k0 = {bit_array_to_string(k0)} (j={j0:2d}, angle={qds.key_to_angle(k0):.4f})   "
            f"k1 = {bit_array_to_string(k1)} (j={j1:2d}, angle={qds.key_to_angle(k1):.4f})"
        )
    print("-" * 72)

total_circuits = len(MESSAGE_BITS) * M * 2
print(f"Total public-key circuits created: {total_circuits}")


### Inspect One Concrete Public-Key Circuit

Let us choose the first `k0` circuit from the first message position.

We will compute:

- the private key bit string
- the integer $j$
- the angle $j\theta$
- the gate angle $2j\theta$
- the resulting quantum state


In [ ]:
example_k0, _ = private_keys[0][0]
example_circuit = public_keys[0][0][0]

example_key = bit_array_to_string(example_k0)
example_j = qds.key_to_int(example_k0)
example_angle = qds.key_to_angle(example_k0)
example_alpha, example_beta = amplitudes_from_j(example_j, qds.THETA)

print(f"example key k = {example_key}")
print(f"j = int(k, 2) = {example_j}")
print(f"theta = {qds.THETA:.6f}")
print(f"j * theta = {example_angle:.6f}")
print(f"R_Y angle = 2 * j * theta = {2 * example_angle:.6f}")
print(f"|f_k> = {format_state(example_alpha, example_beta, decimals=5)}")

example_circuit.draw("mpl", fold=-1)


---
## 13. Worked Toy Example by Hand


Choose a tiny case:

- $L=2$
- key $k = 10$

Then

$$
j = \operatorname{int}(10, 2) = 2,
\qquad
\theta = \frac{\pi}{2^2} = \frac{\pi}{4}.
$$

So

$$
j\theta = 2 \cdot \frac{\pi}{4} = \frac{\pi}{2}.
$$

Therefore

$$
|f_k\rangle
= \cos\!\left(\frac{\pi}{2}\right)|0\rangle + \sin\!\left(\frac{\pi}{2}\right)|1\rangle
= 0\cdot |0\rangle + 1\cdot |1\rangle
= |1\rangle.
$$

This is a great sanity check because it shows the mapping can land exactly on a basis state in a small example.


In [ ]:
L_toy = 2
theta_toy = np.pi / (2 ** L_toy)
key_toy = "10"

_, j_toy, alpha_toy, beta_toy = amplitudes_from_key(key_toy, theta_toy)

print(f"L = {L_toy}")
print(f"key = {key_toy}")
print(f"j = {j_toy}")
print(f"theta = {theta_toy:.6f}")
print(f"j * theta = {j_toy * theta_toy:.6f}")
print(f"|f_k> = {format_state(alpha_toy, beta_toy, decimals=6)}")


### Compare Two Tiny States

Now compare:

- $k = 10$, so $j=2$
- $k' = 01$, so $j'=1$

The overlap is

$$
\langle f_k|f_{k'}\rangle
= \cos\bigl((2-1)\theta\bigr)
= \cos\!\left(\frac{\pi}{4}\right)
= \frac{1}{\sqrt{2}}.
$$

So these states are neither identical nor orthogonal.


In [ ]:
key_a = "10"
key_b = "01"

_, j_a, alpha_a, beta_a = amplitudes_from_key(key_a, theta_toy)
_, j_b, alpha_b, beta_b = amplitudes_from_key(key_b, theta_toy)

overlap_toy = overlap_from_indices(j_a, j_b, theta_toy)
p_swap_toy = swap_probability_from_overlap(overlap_toy)

print(f"key_a = {key_a}, j_a = {j_a}, state_a = {format_state(alpha_a, beta_a, 6)}")
print(f"key_b = {key_b}, j_b = {j_b}, state_b = {format_state(alpha_b, beta_b, 6)}")
print(f"<a|b> = {overlap_toy:.6f}")
print(f"P(ancilla = 0) in SWAP test = {p_swap_toy:.6f}")


---
## 14. Public-Key Equality Demo with the SWAP Test


The repository demonstrates two cases:

1. compare a state with itself
2. compare two different public-key states

We expect:

- identical states $\Rightarrow P(0)$ very close to $1$
- different states $\Rightarrow P(0)$ smaller than $1$


In [ ]:
result_same = qds.verify_key_pair(idx_b=0, i=0, j=0)

bit0_value = int(MESSAGE_BITS[0])
bit1_value = int(MESSAGE_BITS[1])

circ_bit0 = public_keys[0][0][bit0_value]
circ_bit1 = public_keys[1][0][bit1_value]
p_diff, counts_diff = qds.swap_test(circ_bit0, circ_bit1)

key_bit0 = private_keys[0][0][bit0_value]
key_bit1 = private_keys[1][0][bit1_value]
j_bit0 = qds.key_to_int(key_bit0)
j_bit1 = qds.key_to_int(key_bit1)

overlap_sq = overlap_from_indices(j_bit0, j_bit1, qds.THETA) ** 2
p_theory = (1.0 + overlap_sq) / 2.0

print("SWAP test comparison")
print("=" * 60)
print(f"Identical-state test P(0): {result_same['p_same']:.4f}")
print(f"Different-state test P(0): {p_diff:.4f}")
print(f"Theoretical different-state P(0): {p_theory:.4f}")
print(f"Different keys used: j0 = {j_bit0}, j1 = {j_bit1}")
print(f"|<f_k|f_k'>|^2 = {overlap_sq:.4f}")

labels = ["same state", "different states"]
measured = [result_same["p_same"], p_diff]
theory = [1.0, p_theory]
x = np.arange(len(labels))
width = 0.32

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, measured, width, label="measured", color="#1f77b4")
ax.bar(x + width / 2, theory, width, label="theory", color="#ff7f0e", alpha=0.75)
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1.5, label="orthogonal lower limit")
ax.axhline(QuantumDigitalSignature.IDENTITY_THRESHOLD, color="green", linestyle="--", linewidth=1.5, label="identity threshold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("P(ancilla = 0)")
ax.set_title("SWAP test: identical versus different states")
ax.legend()
plt.show()


### Read the Plot Carefully

The first bar should sit essentially at $1$ because comparing a circuit with itself means the overlap is exactly $1$.

The second bar depends on the actual random keys generated for this run.
Its value is controlled by the overlap

$$
|\langle f_k|f_{k'}\rangle|^2.
$$

This is our first concrete bridge between:

- the trigonometric formula
- the quantum circuit
- the simulated measurement statistics
